In [ ]:
from selenium import webdriver
from selenium.webdriver.common.keys import Keys
import pandas as pd

df_produtos = pd.read_excel("Produtos.xlsx")
navegador = webdriver.Chrome()

In [ ]:
def pesquisa_gutenberg(nome, autor, navegador):
    lista_palavras_autor = autor.split(" ")
    navegador.get("https://gutenberg.org/")
    busca = navegador.find_element("class name", "search-input")
    busca.send_keys(nome)
    busca.send_keys(Keys.ENTER)
    lista_resultados = navegador.find_elements("class name", "booklink")
    link = None
    for resultado in lista_resultados:
        texto = resultado.text
        if nome.lower() in texto.lower():
            if all(palavra.lower() in texto.lower() for palavra in lista_palavras_autor):
                link = resultado.find_element("class name", "link").get_attribute("href")
                break

    if link:
        preco = 0
    else:
        preco = None
    
    return link, preco

def pesquisa_bookstoscrape(nome, categoria, navegador):
    navegador.get("https://books.toscrape.com/") 
    lista_categorias = navegador.find_element(By.CSS_SELECTOR, "ul.nav.nav-list")
    try:
        lista_categorias.find_element("link text", categoria).click()
    except:
        print("Categoria não encontrada")

    link = None
    preco = None
    encontrado = False
    while not encontrado:
        lista_resultados = navegador.find_elements("class name", "product_pod")
        for resultado in lista_resultados:
            elemento_h3 = resultado.find_element("tag name", "h3")
            elemento_a = elemento_h3.find_element("tag name", "a")
            titulo = elemento_a.get_attribute("title")
            if nome.lower() in titulo.lower():
                encontrado = True
                link = elemento_a.get_attribute("href")
                preco = resultado.find_element("class name", "price_color").text
                break
        try:
            navegador.find_element("link text", "next").click()
        except:
            break
    return link, preco

df_produtos["link"] = ""
for linha in df_produtos.index:
    nome = df_produtos.loc[linha,"nome"]
    autor = df_produtos.loc[linha,"autor"]
    categoria = df_produtos.loc[linha,"categoria"]

    link1, preco1 = pesquisa_gutenberg(nome, autor, navegador)
    if not link1:
        link2, preco2 = pesquisa_bookstoscrape(nome, categoria, navegador)
        df_produtos.loc[linha, "preco"] = preco2
        df_produtos.loc[linha, "link"] = link2
    else:
        df_produtos.loc[linha, "preco"] = preco1
        df_produtos.loc[linha, "link"] = link1

In [ ]:
df_produtos.to_excel("ProdutosAtualizado.xlsx", index = False)
navegador.quit()

In [13]:
from selenium import webdriver
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

from selenium.common.exceptions import (
    TimeoutException,
    NoSuchElementException
)

import pandas as pd


df_produtos = pd.read_excel("Produtos.xlsx")

df_produtos["link"] = ""
df_produtos["preco"] = None

navegador = webdriver.Chrome()

WAIT = 10

In [14]:
def pesquisa_gutenberg(nome, autor, navegador):
    try:
        navegador.get("https://gutenberg.org/")

        busca = WebDriverWait(navegador, WAIT).until(
            EC.presence_of_element_located(
                (By.CLASS_NAME, "search-input")
            )
        )

        busca.clear()
        busca.send_keys(str(nome))
        busca.send_keys(Keys.ENTER)

        WebDriverWait(navegador, WAIT).until(
            EC.presence_of_element_located(
                (By.CLASS_NAME, "booklink")
            )
        )

        lista_resultados = navegador.find_elements(
            By.CLASS_NAME,
            "booklink"
        )

        lista_palavras_autor = str(autor).split()

        for resultado in lista_resultados:
            texto = resultado.text.lower()

            if str(nome).lower() in texto:

                if all(
                    palavra.lower() in texto
                    for palavra in lista_palavras_autor
                ):
                    link = resultado.find_element(
                        By.CLASS_NAME,
                        "link"
                    ).get_attribute("href")

                    return link, 0

    except Exception as erro:
        print(f"Erro Gutenberg ({nome}): {erro}")

    return None, None


for linha in df_produtos.index:

    nome = df_produtos.loc[linha, "nome"]
    autor = df_produtos.loc[linha, "autor"]
    categoria = df_produtos.loc[linha, "categoria"]

    print(f"Processando: {nome}")

    link, preco = pesquisa_gutenberg(
        nome,
        autor,
        navegador
    )

    df_produtos.loc[linha, "link"] = link
    df_produtos.loc[linha, "preco"] = preco


df_produtos.to_excel(
    "ProdutosAtualizado.xlsx",
    index=False
)

navegador.quit()

print("Concluído!")

Processando: Frankenstein
Processando: Romeo and Juliet
Processando: The Great Gatsby
Concluído!
